# 3D Segmentation with MicroAtlas

This notebook demonstrates **MicroAtlas** for 3D cell segmentation on the **HBEC cell aggregate** dataset.

MicroAtlas (built on Cellpose-SAM) supports two 3D segmentation modes:
- **`do_3D=True`** — true 3D segmentation using 3D convolution kernels, capturing full volumetric context
- **`stitch_threshold`** — 2D slice-by-slice segmentation followed by 3D stitching based on IoU overlap across slices

We use the HBEC cell aggregate (a deconvolved multi-channel 3D volume) as a representative example to visualize and compare both modes.

## Data

The **HBEC cell aggregate** dataset is a multi-channel 3D deconvolved fluorescence image of a human bronchial epithelial cell aggregate.

**Download:** [Sealife_deconv_data.tif](https://drive.google.com/file/d/1_v42S2yDxZ_H4jNXQOhT3Ml5OCROaRG2/view?usp=sharing)

After downloading, place the file in `3d_image/Sealife_deconv_data.tif`.

The volume contains multiple channels (e.g. nuclei, cytoplasm) across ~500 Z-slices.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Ensure src/ is on the import path
SRC_DIR = Path.cwd().parent / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cellpose import io, models, utils, plot
import torch
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device('cuda')
io.logger_setup()

In [ ]:
# Path to data (local to this notebook)
DATA_PATH = Path('./3d_image/Sealife_deconv_data.tif')

img_3d = io.imread(str(DATA_PATH))
print(f"shape : {img_3d.shape}")
print(f"dtype : {img_3d.dtype}")
print(f"range : [{img_3d.min()}, {img_3d.max()}]")

# Show a few representative Z slices
n_z = img_3d.shape[0]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
z_indices = np.linspace(0, n_z - 1, 8, dtype=int)
for col, z in enumerate(z_indices):
    slc = img_3d[z]
    # normalize per channel for display
    rgb = slc.copy().astype(float)
    if rgb.ndim == 3 and rgb.shape[0] <= 4:
        rgb = rgb.transpose(1, 2, 0)
    for c in range(rgb.shape[-1] if rgb.ndim == 3 else 1):
        ch = rgb[..., c] if rgb.ndim == 3 else rgb
        lo, hi = np.percentile(ch[ch > 0], [1, 99]) if (ch > 0).any() else (0, 1)
        rgb[..., c] = np.clip((ch - lo) / max(hi - lo, 1e-10), 0, 1) if rgb.ndim == 3 else np.clip((ch - lo) / max(hi - lo, 1e-10), 0, 1)
    axes[0, col].imshow(rgb if rgb.ndim == 3 else rgb, cmap="gray")
    axes[0, col].set_title(f"Z={z}")
    axes[0, col].axis("off")
    axes[0, col].set_xlabel("raw" if col == 0 else "")

# Show individual channels for middle slice
mid_z = n_z // 2
n_ch = img_3d.shape[-3] if img_3d.ndim == 4 else 1
ch_names = ['Channel 0', 'Channel 1', 'Channel 2', 'Channel 3']
for c in range(min(n_ch, 4)):
    ch_slc = img_3d[mid_z, c] if img_3d.ndim == 4 else img_3d[mid_z]
    axes[1, c].imshow(ch_slc, cmap="gray")
    axes[1, c].set_title(f"{ch_names[c]} (Z={mid_z})")
    axes[1, c].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Load MicroAtlas model
model = models.CellposeModel(gpu=True, pretrained_model='./microatlas/microatlas')
model.net.eval()
print(f'MicroAtlas model loaded on {device}')

# Infer axes from data shape
# img_3d shape: (Z, C, Y, X) -> z_axis=0, channel_axis=1
z_axis = 0
channel_axis = 1
print(f"Data shape: {img_3d.shape}, z_axis={z_axis}, channel_axis={channel_axis}")

In [ ]:
# ---- do_3D: true 3D segmentation ----
print("Running do_3D segmentation...")
tic = time.time()
masks_do3d, flows_do3d, styles_do3d = model.eval(
    img_3d, z_axis=z_axis, channel_axis=channel_axis,
    batch_size=32, do_3D=True,
    flow3D_smooth=1,
    diameter=None, anisotropy=None,
    niter=None, flow_threshold=0.4,
    cellprob_threshold=0.0, compute_masks=True)
time_do3d = time.time() - tic
n_cells_do3d = masks_do3d.max()
print(f"do_3D: {n_cells_do3d} cells detected | shape={masks_do3d.shape} | time={time_do3d:.2f}s")

In [ ]:
# ---- stitch: 2D slices + 3D stitching ----
print("Running stitch segmentation...")
tic = time.time()
masks_stitch, flows_stitch, styles_stitch = model.eval(
    img_3d, z_axis=z_axis, channel_axis=channel_axis,
    batch_size=32, do_3D=False,
    stitch_threshold=0.5,
    diameter=None, anisotropy=None,
    niter=None, flow_threshold=0.4,
    cellprob_threshold=0.0, compute_masks=True)
time_stitch = time.time() - tic
n_cells_stitch = masks_stitch.max()
print(f"stitch: {n_cells_stitch} cells detected | shape={masks_stitch.shape} | time={time_stitch:.2f}s")

In [ ]:
# Visualize segmentation results on representative Z slices
masks_dict = {"do_3D": masks_do3d, "stitch": masks_stitch}
n_z = img_3d.shape[z_axis]
slice_indices = np.linspace(0, n_z - 1, 8, dtype=int)
method_names = list(masks_dict.keys())

fig, axes = plt.subplots(len(method_names) + 1, len(slice_indices),
                         figsize=(3.5 * len(slice_indices), 3.5 * (len(method_names) + 1)))

for col, z_idx in enumerate(slice_indices):
    # Raw slice
    raw_slice = np.take(img_3d, z_idx, axis=z_axis)
    rgb_img = raw_slice.copy().astype(float)
    if rgb_img.ndim == 3 and rgb_img.shape[0] <= 4:
        rgb_img = rgb_img.transpose(1, 2, 0)
    for c in range(rgb_img.shape[-1] if rgb_img.ndim == 3 else 1):
        ch = rgb_img[..., c] if rgb_img.ndim == 3 else rgb_img
        lo, hi = np.percentile(ch[ch > 0], [1, 99]) if (ch > 0).any() else (0, 1)
        rgb_img[..., c] = np.clip((ch - lo) / max(hi - lo, 1e-10), 0, 1)
    axes[0, col].imshow(rgb_img)
    axes[0, col].set_title(f"Z={z_idx}", fontsize=10)
    axes[0, col].axis("off")

    # Overlay for each method
    for row, method_name in enumerate(method_names):
        ax = axes[row + 1, col]
        mask_slice = np.take(masks_dict[method_name], z_idx, axis=z_axis)
        overlay = plot.outline_view(rgb_img, mask_slice, color=[1, 0, 0])
        ax.imshow(overlay)
        if col == 0:
            ax.set_ylabel(method_name, fontsize=12, fontweight='bold')
        ax.axis("off")

plt.suptitle("2D Slice Overlay: Raw | do_3D | stitch", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from skimage.measure import marching_cubes

def render_3d_surface_combined(mask, method_name="", downsample=4, alpha=0.6):
    """Render the overall isosurface of a 3D mask volume."""
    print(f"  Extracting isosurface (downsample={downsample})...")
    if downsample > 1:
        mask_ds = mask[::downsample, ::downsample, ::downsample]
    else:
        mask_ds = mask

    verts, faces, normals, _ = marching_cubes(
        (mask_ds > 0).astype(np.float32),
        level=0.5,
        spacing=(downsample,) * 3,
    )

    fig = plt.figure(figsize=(10, 9))
    ax = fig.add_subplot(111, projection="3d")
    mesh = Poly3DCollection(verts[faces])
    mesh.set_facecolor((0.15, 0.55, 0.85, alpha))
    mesh.set_edgecolor("none")
    ax.add_collection3d(mesh)

    ax.set_xlim(verts[:, 0].min(), verts[:, 0].max())
    ax.set_ylim(verts[:, 1].min(), verts[:, 1].max())
    ax.set_zlim(verts[:, 2].min(), verts[:, 2].max())
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(f"{method_name} — isosurface ({mask.max()} cells)", fontsize=13)
    ax.view_init(elev=25, azim=-60)
    plt.tight_layout()
    plt.show()

# Render both modes side by side
fig, axes = plt.subplots(1, 2, figsize=(20, 9), subplot_kw={"projection": "3d"})

for ax, (name, mask_vol) in zip(axes, [("do_3D", masks_do3d), ("stitch", masks_stitch)]):
    mask_ds = mask_vol[::4, ::4, ::4]
    verts, faces, _, _ = marching_cubes(
        (mask_ds > 0).astype(np.float32), level=0.5, spacing=(4,) * 3)
    mesh = Poly3DCollection(verts[faces])
    mesh.set_facecolor((0.15, 0.55, 0.85, 0.6))
    mesh.set_edgecolor("none")
    ax.add_collection3d(mesh)
    ax.set_xlim(verts[:, 0].min(), verts[:, 0].max())
    ax.set_ylim(verts[:, 1].min(), verts[:, 1].max())
    ax.set_zlim(verts[:, 2].min(), verts[:, 2].max())
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(f"{name} ({mask_vol.max()} cells)", fontsize=13)
    ax.view_init(elev=25, azim=-60)

plt.suptitle("3D Isosurface — All Cells", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def render_3d_surface_cells(mask, method_name="", max_cells=30, downsample=2, alpha=0.7):
    """Render individual cell surfaces with random colors."""
    labels, counts = np.unique(mask, return_counts=True)
    labels, counts = labels[1:], counts[1:]  # remove background
    order = np.argsort(counts)[::-1][:max_cells]
    top_labels = labels[order]

    if downsample > 1:
        mask_ds = mask[::downsample, ::downsample, ::downsample]
        sf = downsample
    else:
        mask_ds = mask
        sf = 1

    np.random.seed(42)
    colors = np.random.rand(len(top_labels), 3)

    fig = plt.figure(figsize=(11, 9))
    ax = fig.add_subplot(111, projection="3d")

    for i, lbl in enumerate(top_labels):
        cell_bin = (mask_ds == lbl).astype(np.float32)
        if cell_bin.sum() < 10:
            continue
        try:
            verts, faces, _, _ = marching_cubes(cell_bin, level=0.5, spacing=(sf,) * 3)
        except Exception:
            continue
        mesh = Poly3DCollection(verts[faces])
        mesh.set_facecolor((*colors[i], alpha))
        mesh.set_edgecolor("none")
        ax.add_collection3d(mesh)

    shape = mask_ds.shape
    ax.set_xlim(0, shape[2] * sf)
    ax.set_ylim(0, shape[1] * sf)
    ax.set_zlim(0, shape[0] * sf)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(f"{method_name} — Top {len(top_labels)} cells", fontsize=13)
    ax.view_init(elev=25, azim=-60)
    plt.tight_layout()
    plt.show()

# Per-cell rendering for both modes
fig, axes = plt.subplots(1, 2, figsize=(22, 9), subplot_kw={"projection": "3d"})

for ax, (name, mask_vol) in zip(axes, [("do_3D", masks_do3d), ("stitch", masks_stitch)]):
    labels, counts = np.unique(mask_vol, return_counts=True)
    labels, counts = labels[1:], counts[1:]
    order = np.argsort(counts)[::-1][:30]
    top_labels = labels[order]

    mask_ds = mask_vol[::2, ::2, ::2]
    np.random.seed(42)
    colors = np.random.rand(len(top_labels), 3)

    for i, lbl in enumerate(top_labels):
        cell_bin = (mask_ds == lbl).astype(np.float32)
        if cell_bin.sum() < 10:
            continue
        try:
            verts, faces, _, _ = marching_cubes(cell_bin, level=0.5, spacing=(2,) * 3)
        except Exception:
            continue
        mesh = Poly3DCollection(verts[faces])
        mesh.set_facecolor((*colors[i], 0.7))
        mesh.set_edgecolor("none")
        ax.add_collection3d(mesh)

    shape = mask_ds.shape
    ax.set_xlim(0, shape[2] * 2)
    ax.set_ylim(0, shape[1] * 2)
    ax.set_zlim(0, shape[0] * 2)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(f"{name} — Top 30 cells", fontsize=13)
    ax.view_init(elev=25, azim=-60)

plt.suptitle("3D Per-Cell Rendering", fontsize=14)
plt.tight_layout()
plt.show()

## Results Summary

Comparison of the two 3D segmentation modes on the HBEC cell aggregate dataset.

In [ ]:
# Summary table
print(f"{'Mode':<12s} {'# Cells':>8s} {'Time (s)':>10s} {'Mask Shape'}")
print("-" * 50)
print(f"{'do_3D':<12s} {masks_do3d.max():>8d} {time_do3d:>10.2f} {str(masks_do3d.shape)}")
print(f"{'stitch':<12s} {masks_stitch.max():>8d} {time_stitch:>10.2f} {str(masks_stitch.shape)}")

print()
print("Key differences:")
print("  do_3D    — Uses full 3D context via 3D convolutions; better for isotropic or thick volumes")
print("  stitch   — Segments each 2D slice independently then stitches across Z; faster, works well for thin slices")